# A complete study

This lesson runs one fresh table through the whole path: the sums, the line, the decomposition, the standard errors, a test, a prediction interval, and a comparison with a model that refuses the slope. A sampling experiment then checks the coverage the $t$ interval claims.


## The table

$$
\begin{array}{c|cccc}
x & 0 & 1 & 2 & 3 \\
\hline
y & 3 & 4 & 7 & 12
\end{array}
$$

The residuals below were built so that the least-squares line would be exactly $2 + 3x$, and the arithmetic is required to discover that line rather than to assume it.

$$
\begin{aligned}
n &= 4, & \sum x &= 6, & \sum y &= 26, \\
\sum xy &= 0\cdot 3 + 1\cdot 4 + 2\cdot 7 + 3\cdot 12 = 54, \\
\sum x^2 &= 0 + 1 + 4 + 9 = 14.
\end{aligned}
$$

$$
\begin{aligned}
b_1 &= \dfrac{4\cdot 54 - 6\cdot 26}{4\cdot 14 - 36} = \dfrac{216 - 156}{20} = 3, \\
b_0 &= \dfrac{26 - 3\cdot 6}{4} = 2.
\end{aligned}
$$

Fitted values: $2, 5, 8, 11$. Residuals: $1, -1, -1, 1$. Then $\mathrm{RSS} = 4$, $\bar y = 13/2$, and

$$
\mathrm{TSS} = \left(-\dfrac{7}{2}\right)^2 + \left(-\dfrac{5}{2}\right)^2 + \left(\dfrac{1}{2}\right)^2 + \left(\dfrac{11}{2}\right)^2 = \dfrac{49 + 25 + 1 + 121}{4} = 49.
$$

So $\mathrm{SSR} = 45$ and $R^2 = 45/49$. Also $S_{xx} = 5$ and $S_{xy} = 15$, which return the same slope $3$.


In [1]:
# Every sum on the four-point study, then R-squared.
from fractions import Fraction
x = [0, 1, 2, 3]
y = [3, 4, 7, 12]
n = 4
sx, sy = sum(x), sum(y)
sxy = sum(a * b for a, b in zip(x, y))
sx2 = sum(a * a for a in x)
slope = Fraction(n * sxy - sx * sy, n * sx2 - sx ** 2)
intercept = Fraction(sy - slope * sx, n)
fitted = [intercept + slope * xi for xi in x]
errors = [yi - yh for yi, yh in zip(y, fitted)]
mean = Fraction(sy, n)
tss = sum((yi - mean) ** 2 for yi in y)
rss = sum(error ** 2 for error in errors)
print("line", intercept, slope)
print("residuals", errors)
print("TSS RSS R2", tss, rss, 1 - rss / tss)
assert intercept == 2 and slope == 3
assert errors == [1, -1, -1, 1]
assert tss == 49 and rss == 4
assert 1 - rss / tss == Fraction(45, 49)


line 2 3
residuals [Fraction(1, 1), Fraction(-1, 1), Fraction(-1, 1), Fraction(1, 1)]
TSS RSS R2 49 4 45/49


## Standard error, test, and a prediction

$n - p = 2$ and $\mathrm{RSS} = 4$, so $s^2 = 2$. With $S_{xx} = 5$,

$$
\operatorname{se}(b_1) = \sqrt{\dfrac{2}{5}}, \qquad
t = \dfrac{3}{\sqrt{2/5}} = 3\sqrt{\dfrac{5}{2}}.
$$

The square is $F = 45/2 = 22.5$, the comparison of $\mathrm{SSR} = 45$ on one degree of freedom with $\mathrm{RSS}/2 = 2$. The $0.975$ quantile of $t_2$ is approximately $4.303$. The statistic $3\sqrt{5/2} \approx 4.743$ is larger, so a two-sided $5\%$ test rejects a zero slope. The interval is

$$
3 \pm t_{2,\,0.975}\sqrt{\dfrac{2}{5}}.
$$

At $x_0 = 4$, one step past the last observation, the point prediction is $2 + 12 = 14$. The prediction standard error is

$$
s\sqrt{1 + \dfrac{1}{4} + \dfrac{(4 - 3/2)^2}{5}}
= \sqrt{2}\sqrt{\dfrac{5}{2}} = \sqrt{5},
$$

because $1 + 1/4 + 6.25/5 = 2.5$. The $95\%$ prediction interval is $14 \pm t_{2,\,0.975}\sqrt{5}$.


In [2]:
# The t statistic, the F statistic, and the prediction interval at x=4.
import math
import scipy.stats as stats
se = math.sqrt(2 / 5)
t_stat = 3 / se
f_stat = (45 / 1) / (4 / 2)
critical = stats.t.ppf(0.975, 2)
lower, upper = 3 - critical * se, 3 + critical * se
prediction = 14
se_pred = math.sqrt(5)
pred_interval = (prediction - critical * se_pred, prediction + critical * se_pred)
print("t", t_stat, "F", f_stat, "critical", critical)
print("slope interval", lower, upper)
print("prediction interval", pred_interval)
assert abs(t_stat - 3 * math.sqrt(5 / 2)) < 1e-12
assert abs(f_stat - 22.5) < 1e-12
assert abs(t_stat ** 2 - f_stat) < 1e-9
assert t_stat > critical and lower > 0
assert abs(se_pred - math.sqrt(5)) < 1e-12


t 4.743416490252569 F 22.5 critical 4.302652729749462
slope interval 0.27876347865015605 5.721236521349844
prediction interval (np.float64(4.378976012705172), np.float64(23.621023987294826))


## A model that refuses the slope

The intercept-only fit predicts $\bar y = 13/2$ at every $x$, including at $x = 4$. Its residual sum of squares on the four training rows is $\mathrm{TSS} = 49$. The line's residual sum of squares is $4$.

On the single new location $x = 4$, suppose the response that actually arrives is the noise-free value of the line that generated the lesson, $2 + 3\cdot 4 = 14$. The line predicts $14$ and the constant predicts $6.5$. The squared prediction errors are $0$ and $(14 - 13/2)^2 = 225/4$. This is one invented check point, not a proof of future performance. It shows how the training residual sums, $4$ against $49$, and a prediction error can be reported side by side.


In [3]:
# The constant model fits the training rows worse and misses x=4 by 7.5.
from fractions import Fraction
training_rss_line = 4
training_rss_constant = 49
line_error = 14 - 14
constant_error = 14 - Fraction(13, 2)
print("training RSS", training_rss_line, training_rss_constant)
print("squared errors at x=4", line_error ** 2, constant_error ** 2)
assert training_rss_line < training_rss_constant
assert constant_error ** 2 == Fraction(225, 4)


training RSS 4 49
squared errors at x=4 0 225/4


## Coverage of the slope interval

The $t$ interval claims a $95\%$ coverage probability when the Gauss–Markov model holds and the errors are normal. Check the claim on the five-point design $x = 1, 2, 3, 4, 5$, where $S_{xx} = 10$ and the true line is $1 + 2x$, with independent standard normal errors. For each of $8{,}000$ samples, form

$$
b_1 \pm t_{3,\,0.975}\,\dfrac{s}{\sqrt{S_{xx}}}, \qquad s^2 = \dfrac{\mathrm{RSS}}{3},
$$

and record whether the interval contains $2$. The sampling standard deviation of a proportion near $0.95$ on $8{,}000$ draws is about $0.0024$, so the observed coverage should fall in a band around $0.95$. This does not replace the proof. It asks whether the code's interval is the interval the proof describes.


In [4]:
# Empirical coverage of the 95 percent t interval for the slope.
import numpy as np
import scipy.stats as stats
rng = np.random.default_rng(1)
x = np.array([1.0, 2, 3, 4, 5])
xbar = x.mean()
sxx = np.sum((x - xbar) ** 2)
draws = 8000
y = 1 + 2 * x + rng.normal(0.0, 1.0, size=(draws, x.size))
ybar = y.mean(axis=1)
slope = ((y - ybar[:, None]) @ (x - xbar)) / sxx
intercept = ybar - slope * xbar
residuals = y - intercept[:, None] - slope[:, None] * x
s = np.sqrt(np.sum(residuals ** 2, axis=1) / 3)
half = stats.t.ppf(0.975, 3) * s / np.sqrt(sxx)
coverage = np.mean((slope - half <= 2) & (2 <= slope + half))
print("coverage", coverage)
print("mean slope", slope.mean())
assert 0.935 < coverage < 0.965
assert abs(slope.mean() - 2) < 0.02


coverage 0.9505
mean slope 2.01109242391534


## Pitfall

The four-point study rejected a zero slope and produced $R^2 = 45/49$ from four observations and two residual degrees of freedom. That is a complete calculation on a tiny table. It is not a stable description of a population. The coverage experiment needed thousands of repeated samples before $95\%$ could be seen as a frequency. One table, however carefully fitted, is one draw.


In [5]:
# Two residual degrees of freedom are enough for the formulas and thin as evidence.
n, p = 4, 2
print("residual degrees of freedom", n - p)
assert n - p == 2


residual degrees of freedom 2


## Summary

- The study table fits $2 + 3x$, with residuals $1, -1, -1, 1$, $\mathrm{RSS} = 4$, and $R^2 = 45/49$.
- $t = 3\sqrt{5/2}$ and $F = 45/2$. At $x = 4$ the prediction standard error is $\sqrt{5}$.
- Under the normal model on the five-point design, the $95\%$ $t$ interval for the slope covers $2$ in about $95\%$ of repeated samples.
